### librerias y paquetes necesarios

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pymcel as pym 
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

Bienvenido a PyMCel v0.9.17 ¡al infinito y más allá!


**Se consultan las respectivas efemerides de las lunas mas importantes de jupiter con `consulta_horizons`** de la s lunas (*Mimas-Tethys, Enceladus-Dione and Titan-Hyperion*)


In [ ]:
_, _, X_m = pym.consulta_horizons(id ='Mimas', location = '@6', epochs= '2024-01-01 00:00:00')
_, _, X_e = pym.consulta_horizons(id ='Enceladus', location = '@6', epochs= '2024-01-01 00:00:00')
_, _, X_t = pym.consulta_horizons(id ='Tethys', location = '@6', epochs= '2024-01-01 00:00:00')
_, _, X_d = pym.consulta_horizons(id ='Dione', location = '@6', epochs= '2024-01-01 00:00:00')
_, _, X_r = pym.consulta_horizons(id ='606', location = '@6', epochs= '2024-01-01 00:00:00')
_, _, X_J = pym.consulta_horizons(id ='599', location = '@6', epochs= '2024-01-01 00:00:00')
_, _, X_S = pym.consulta_horizons(id ='699', location = '@5', epochs= '2024-01-01 00:00:00')
_, _, X_U = pym.consulta_horizons(id ='799', location = '@5', epochs= '2024-01-01 00:00:00')
_, _, X_N = pym.consulta_horizons(id ='Hyperion', location = '@5', epochs= '2024-01-01 00:00:00')
_, _, X_sol = pym.consulta_horizons(id ='Sun', location = '@5', epochs= '2024-01-01 00:00:00') 


In [3]:
# Construye un DataFrame con posicion y velocidad (SI)
state_vectors = {
    "Mimas": X_m,
    "Enceladus": X_e,
    "Tethys": X_t,
    "Dione": X_d,
    "Rhea": X_r,
    "Jupiter": X_J,
    "Saturn": X_S,
    "Uranus": X_U,
    "Hyperion": X_N,
    "Sun": X_sol,
}

rows = []
for body, vec in state_vectors.items():
    arr = np.asarray(vec)
    if arr.ndim == 1 and arr.size == 6:
        arr = arr.reshape(1, 6)
    if arr.ndim != 2 or arr.shape[1] != 6:
        raise ValueError(f"Vector de estado inesperado para {body}: shape {arr.shape}")
    for i, (x, y, z, vx, vy, vz) in enumerate(arr):
        rows.append({
            "body": body,
            "row": i,
            "x": x,
            "y": y,
            "z": z,
            "vx": vx,
            "vy": vy,
            "vz": vz,
        })

df_state = pd.DataFrame(rows)
df_state

,body,row,x,y,z,vx,vy,vz
0,Mimas,0,-3.191482e+07,-1.611224e+08,9.323857e+07,13851.516653,-2522.915247,72.239696
1,Enceladus,0,2.325774e+08,2.552997e+07,-3.596181e+07,-2121.589692,11164.699962,-5643.072296
2,Tethys,0,2.113037e+08,1.698986e+08,-1.145448e+08,-7811.372186,7633.608875,-3093.863090
3,Dione,0,-3.605384e+08,-8.395682e+07,7.897470e+07,2926.973136,-8592.528468,4213.331230
4,Rhea,0,9.236158e+08,6.387412e+08,-4.214333e+08,-3486.128786,4109.701546,-1772.144669
5,Jupiter,0,-8.232226e+11,1.087756e+12,2.999200e+10,-12626.681862,863.034304,451.793617
6,Saturn,0,8.232225e+11,-1.087756e+12,-2.999189e+10,12625.844461,-863.896616,-451.387906
7,Uranus,0,1.313143e+12,1.757064e+12,-1.397889e+09,4107.713980,-5827.445232,-87.220534
8,Hyperion,0,8.240131e+11,-1.086577e+12,-3.065841e+10,8766.839235,1757.768322,-1401.705767
9,Sun,0,-5.225708e+11,-5.318268e+11,1.390074e+10,9479.547148,-9781.812772,-171.449677


### Definimos coordenadas canónicas 

- $U_L = \vec{r}_S - \vec{r}_J$
- $U_M = M_S$
- $U_T = \sqrt{\frac{U_L^3}{U_M\, G}}$

In [22]:
X_S[0:3]

array([ 8.23222462e+11, -1.08775623e+12, -2.99918894e+10])

In [17]:
G = 6.67430e-11  # Constante de gravitación universal en m^3 kg^-1 s^-2
M_S = 5.683e26  # Masa de Saturno en kg
U_L = np.linalg.norm(X_S[:3] - X_J[:3], axis=0)
U_M = M_S
U_T = np.sqrt(U_L**3 / (G * U_M))
print(f"Unidad de tiempo (U_T): {U_T} segundos")
print(f"Unidad de longitud (U_L): {U_L} metros")
print(f"Unidad de masa (U_M): {U_M} kg")

Unidad de tiempo (U_T): 23147523811.46762 segundos
Unidad de longitud (U_L): 2728961987182.351 metros
Unidad de masa (U_M): 5.683e+26 kg


Con esto, podemos cambiar el datframe a unidades canonicas:

### Consulta de Masas desde JPL Horizons

A continuación, se consultan las masas de los cuerpos definidos anteriormente directamente desde el sistema Horizons de JPL.
**Nota:** Se ha detectado que el ID '606' utilizado para "Rhea" corresponde en realidad a **Titán**. Rhea es usualmente el ID '605'. Los valores mostrados abajo corresponden al ID consultado.

In [5]:
import requests
import re
import pandas as pd

# Diccionario de cuerpos e IDs usados en el notebook
bodies_ids = {
    'Mimas': 'Mimas',
    'Enceladus': 'Enceladus',
    'Tethys': 'Tethys',
    'Dione': 'Dione',
    'Rhea (ID en código: 606)': '606', # Nota: 606 es Titan
    'Jupiter': '599',
    'Saturn': '699',
    'Uranus': '799',
    'Hyperion': 'Hyperion',
    'Sun': 'Sun'
}

def get_mass_si(command_id):
    """Consulta la masa o GM desde JPL Horizons y devuelve la masa en kg."""
    url = 'https://ssd.jpl.nasa.gov/api/horizons.api'
    params = {'format':'json', 'COMMAND':command_id, 'OBJ_DATA':'YES', 'MAKE_EPHEM':'NO'}
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        text = r.json().get('result', '')
    except Exception as e:
        return None

    # Intentar extraer masa directamente "Mass x 10^n (kg) = val"
    # Formato Júpiter: Mass x 10^26 (kg)
    m1 = re.search(r'Mass\s+x\s+10\^(\d+)\s+\(kg\)\s*=\s*([~]?[\d\.]+)', text)
    if m1: 
        return float(m1.group(2).replace('~','')) * (10**float(m1.group(1)))

    # Formato Lunas: Mass (10^19 kg)
    m2 = re.search(r'Mass\s+\(10\^(\d+)\s+kg\)\s*=\s*([~]?[\d\.]+)', text)
    if m2: 
        return float(m2.group(2).replace('~','')) * (10**float(m2.group(1)))

    # Formato Sol: Mass, 10^24 kg
    m3 = re.search(r'Mass,\s+10\^(\d+)\s+kg\s*=\s*([~]?[\d\.]+)', text)
    if m3: 
        return float(m3.group(2).replace('~','')) * (10**float(m3.group(1)))

    # Si no hay masa explicita, usar GM (km^3/s^2)
    mg = re.search(r'GM\s+\(km\^3/s\^2\)\s*=\s*([~]?[\d\.]+)', text)
    if mg:
        G_approx = 6.67430e-11
        gm_km3s2 = float(mg.group(1).replace('~',''))
        gm_m3s2 = gm_km3s2 * 1e9 
        return gm_m3s2 / G_approx

    return None

data_mass = []
for name, horizon_id in bodies_ids.items():
    mass = get_mass_si(horizon_id)
    data_mass.append({'Cuerpo': name, 'ID Horizons': horizon_id, 'Masa (kg)': mass})

df_mass = pd.DataFrame(data_mass)
# Formato científico para mejor lectura
pd.options.display.float_format = '{:.4e}'.format
df_mass

,Cuerpo,ID Horizons,Masa (kg)
0,Mimas,Mimas,3.7500e+19
1,Enceladus,Enceladus,1.0805e+20
2,Tethys,Tethys,6.1760e+20
3,Dione,Dione,1.0957e+21
4,Rhea (ID en código: 606),606,1.3455e+23
5,Jupiter,599,1.8982e+27
6,Saturn,699,5.6832e+26
7,Uranus,799,8.6810e+25
8,Hyperion,Hyperion,1.0800e+19
9,Sun,Sun,1.9884e+30


In [6]:
df_final = pd.merge(df_state, df_mass, left_on='body', right_on='Cuerpo', how='left')
df_final.drop(columns=['Cuerpo', 'ID Horizons'], inplace=True)
df_final

,body,row,x,y,z,vx,vy,vz,Masa (kg)
0,Mimas,0,-3.1915e+07,-1.6112e+08,9.3239e+07,1.3852e+04,-2.5229e+03,7.2240e+01,3.7500e+19
1,Enceladus,0,2.3258e+08,2.5530e+07,-3.5962e+07,-2.1216e+03,1.1165e+04,-5.6431e+03,1.0805e+20
2,Tethys,0,2.1130e+08,1.6990e+08,-1.1454e+08,-7.8114e+03,7.6336e+03,-3.0939e+03,6.1760e+20
3,Dione,0,-3.6054e+08,-8.3957e+07,7.8975e+07,2.9270e+03,-8.5925e+03,4.2133e+03,1.0957e+21
4,Rhea,0,9.2362e+08,6.3874e+08,-4.2143e+08,-3.4861e+03,4.1097e+03,-1.7721e+03,NaN
5,Jupiter,0,-8.2322e+11,1.0878e+12,2.9992e+10,-1.2627e+04,8.6303e+02,4.5179e+02,1.8982e+27
6,Saturn,0,8.2322e+11,-1.0878e+12,-2.9992e+10,1.2626e+04,-8.6390e+02,-4.5139e+02,5.6832e+26
7,Uranus,0,1.3131e+12,1.7571e+12,-1.3979e+09,4.1077e+03,-5.8274e+03,-8.7221e+01,8.6810e+25
8,Hyperion,0,8.2401e+11,-1.0866e+12,-3.0658e+10,8.7668e+03,1.7578e+03,-1.4017e+03,1.0800e+19
9,Sun,0,-5.2257e+11,-5.3183e+11,1.3901e+10,9.4795e+03,-9.7818e+03,-1.7145e+02,1.9884e+30


Vamos a cambiar las coordenadas del `df_final` a unidades canonicas

In [ ]:
df_canonical = df_final.copy()
df_canonical['x'] /= U_L
df_canonical['y'] /= U_L
df_canonical['z'] /= U_L
df_canonical['vx'] /= (U_L / U_T)
df_canonical['vy'   ] /= (U_L / U_T)
df_canonical['vz'] /= (U_L / U_T    )

df_canonical['Masa (kg)'] /= U_M


Creamos la lista `sistema = [dic(m = ms, r = r2, v = vs)]`

In [8]:
sistema = [dict(m= df_canonical['Masa (kg)'][i], 
                r= np.array([df_canonical['x'][i], df_canonical['y'][i], df_canonical['z'][i]]), 
                v= np.array([df_canonical['vx'][i], df_canonical['vy'][i], df_canonical['vz'][i]]) 
                ) for i in range(len(df_canonical))]

In [9]:
# Se resuelve el sistema con pymcel

T = 10
N = 1000
ts = np.linspace(0, T, N)

rs, vs, rcms, vcms, cons = pym.ncuerpos_solucion(sistema, ts)

In [10]:
rs.shape

(9, 1000, 3)

In [11]:
cons['K']

array([1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
       1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
       1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
       1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
       1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
       1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
       1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
       1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
       1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
       1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
       1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
       1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
       1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
       1.46378844e+13, 1.46378844e+13, 1.46378844e+13, 1.46378844e+13,
      

In [12]:
cons['E']

np.float64(14637884354626.012)

In [13]:
# Gráfico 3D de las órbitas con respecto al centro de masa
fig = go.Figure()

# Añadir las trayectorias de cada cuerpo
for i in range(len(sistema)-1):
    # Trayectoria
    fig.add_trace(go.Scatter3d(
        x=rcms[i, :, 0],
        y=rcms[i, :, 1],
        z=rcms[i, :, 2],
        mode='lines',
        name=f'Cuerpo {i+1} (m={sistema[i]["m"]:.2e})',
        line=dict(width=3)
    ))
    
    # Punto inicial
    fig.add_trace(go.Scatter3d(
        x=[rcms[i, 0, 0]],
        y=[rcms[i, 0, 1]],
        z=[rcms[i, 0, 2]],
        mode='markers',
        name=f'Inicio {i+1}',
        marker=dict(size=8, symbol='circle')
    ))
    
    # Punto final
    fig.add_trace(go.Scatter3d(
        x=[rcms[i, -1, 0]],
        y=[rcms[i, -1, 1]],
        z=[rcms[i, -1, 2]],
        mode='markers',
        name=f'Fin {i+1}',
        marker=dict(size=8, symbol='x')
    ))

# Centro de masa (origen)
fig.add_trace(go.Scatter3d(
    x=[0],
    y=[0],
    z=[0],
    mode='markers',
    name='Centro de Masa',
    marker=dict(size=10, color='black', symbol='diamond')
))

fig.update_layout(
    title='Órbitas con respecto al Centro de Masa',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='cube'
    ),
    width=900,
    height=700
)

fig.show()